In [ ]:
# Import Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np
import os
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
from google.colab import drive


In [ ]:
# Mount Google Drive + Define file paths
drive.mount('/content/drive', force_remount=True)
DATA_DIR = '/content/drive/MyDrive/xg-ids/raw_datasets'
TRAIN_FILE = "nslkdd.txt"

Mounted at /content/drive


In [ ]:
# Column names for the NSL-KDD dataset
column_names = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'class', 'difficulty'
]

# Dictonary maps all the subcatagories to major catagories
subcategory_to_major = {
    'back': 'DoS', 'land': 'DoS', 'neptune': 'DoS', 'pod': 'DoS', 'smurf': 'DoS', 'teardrop': 'DoS', 'apache2': 'DoS', 'mailbomb': 'DoS', 'udpstorm': 'DoS', 'processtable': 'DoS',
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L', 'multihop': 'R2L', 'phf': 'R2L', 'spy': 'R2L', 'warezclient': 'R2L', 'warezmaster': 'R2L',
    'named': 'R2L', 'xsnoop': 'R2L', 'xlock': 'R2L', 'sendmail': 'R2L', 'worm': 'R2L', 'snmpgetattack': 'R2L', 'snmpguess': 'R2L',
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R', 'httptunnel': 'U2R', 'sqlattack': 'U2R', 'xterm': 'U2R', 'ps': 'U2R',
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe', 'satan': 'Probe', 'saint': 'Probe', 'mscan': 'Probe',
    'normal': 'Normal'
}

# Load in the raw nslkdd dataset
nslkdd = pd.read_csv('/content/drive/MyDrive/xg-ids/raw_datasets/nslkdd.txt', header=None, names=column_names)

In [ ]:
# Dropping difficulty column (last column) from NSLKDD Dataset
nslkdd = nslkdd.drop('difficulty', axis=1)
# Displays preview of the NSLKDD dataset after dropping difficulty column
print(f"Total Columns (41 features and 1 class label: {nslkdd.shape[1]}")
print("Preview of the first 5 instances of the NSL-KDD Dataset after dropping difficulty column:")
nslkdd.head()

Total Columns (41 features and 1 class label: 42
Preview of the first 5 instances of the NSL-KDD Dataset after dropping difficulty column:


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,0.0,0.0,0.0,0.0,1.00,0.00,0.00,150,25,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal
1,0,udp,other,SF,146,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,1,0.0,0.0,0.0,0.0,0.08,0.15,0.00,255,1,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal
2,0,tcp,private,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,123,6,1.0,1.0,0.0,0.0,0.05,0.07,0.00,255,26,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune
3,0,tcp,http,SF,232,8153,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,5,5,0.2,0.2,0.0,0.0,1.00,0.00,0.00,30,255,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal
4,0,tcp,http,SF,199,420,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,30,32,0.0,0.0,0.0,0.0,1.00,0.00,0.09,255,255,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal


In [ ]:
# Remove R2L and U2R

# Identify subcategories that belong to U2R or R2L
drop_subcategories = {k for k, v in subcategory_to_major.items() if v in ('U2R', 'R2L')}

# Drop all rows where 'class' contains subcategories belonging to U2R or R2L
nslkdd = nslkdd[~nslkdd['class'].isin(drop_subcategories)].reset_index(drop=True)

print(f"Total instances after dropping U2R + R2L: {nslkdd.shape[0]}")
print("Remaining subcategories:", sorted(nslkdd['class'].unique()))

Total instances after dropping U2R + R2L: 124926
Remaining subcategories: ['back', 'ipsweep', 'land', 'neptune', 'nmap', 'normal', 'pod', 'portsweep', 'satan', 'smurf', 'teardrop']


Split Original nslkdd into 90/10 Train/Validate sets

In [ ]:
# TODO: Stratified split the dataset into train(90)/validate(10) ensuring each subcatagory is represented in train and validate

df_train, df_validate = train_test_split(
    nslkdd,
    test_size=0.10,
    random_state=42,
    stratify=nslkdd['class']   # stratify on subcategory, not major category
)

print(f"Train instances:    {df_train.shape[0]}")
print(f"Validate instances: {df_validate.shape[0]}")

# Verify subcategory representation in both splits
print("\nSubcategory distribution in train:")
print(df_train['class'].value_counts())
print("\nSubcategory distribution in validate:")
print(df_validate['class'].value_counts())

# Save the train dataset
df_train.to_csv('/content/drive/MyDrive/xg-ids/3class/datasets/3class_TRAIN.txt', index=False)

# Save the validate dataset
df_validate.to_csv('/content/drive/MyDrive/xg-ids/3class/datasets/3class_VALIDATE.txt', index=False)

Train instances:    112433
Validate instances: 12493

Subcategory distribution in train:
class
normal       60609
neptune      37092
satan         3270
ipsweep       3239
portsweep     2638
smurf         2381
nmap          1344
back           860
teardrop       803
pod            181
land            16
Name: count, dtype: int64

Subcategory distribution in validate:
class
normal       6734
neptune      4122
satan         363
ipsweep       360
portsweep     293
smurf         265
nmap          149
back           96
teardrop       89
pod            20
land            2
Name: count, dtype: int64


#~~~~~ VALIDATE DATASET PROCESSING ~~~~~

In [ ]:

# Load in the validate dataset (with headers)
validate_nslkdd = pd.read_csv('/content/drive/MyDrive/xg-ids/3class/datasets/3class_VALIDATE.txt')

# Displays preview of the validate dataset
print(f"Total instances: {validate_nslkdd.shape[0]}")
print(f"Total Columns (41 features, 1 class label): {validate_nslkdd.shape[1]}")
print("Preview of the first 5 instances of the Validate Dataset before processing:")
validate_nslkdd.head()

Total instances: 12493
Total Columns (41 features, 1 class label): 42
Preview of the first 5 instances of the Validate Dataset before processing:


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class
0,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0.0,0.0,1.0,1.0,1.00,0.00,0.00,27,2,0.04,1.00,0.04,1.00,0.0,0.0,0.96,1.0,ipsweep
1,0,tcp,http,SF,326,328,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,5,5,0.0,0.0,0.0,0.0,1.00,0.00,0.00,5,255,1.00,0.00,0.20,0.04,0.0,0.0,0.00,0.0,normal
2,0,tcp,nnsp,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,127,5,1.0,1.0,0.0,0.0,0.04,0.07,0.00,255,5,0.02,0.07,0.00,0.00,1.0,1.0,0.00,0.0,neptune
3,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,222,11,0.0,0.0,1.0,1.0,0.05,0.06,0.00,255,11,0.04,0.06,0.00,0.00,0.0,0.0,1.00,1.0,neptune
4,0,udp,domain_u,SF,42,42,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,134,214,0.0,0.0,0.0,0.0,1.00,0.00,0.01,255,255,1.00,0.00,0.01,0.00,0.0,0.0,0.00,0.0,normal


In [ ]:
# Rename subcategories to major categories
validate_nslkdd['class'] = validate_nslkdd['class'].map(subcategory_to_major)


# Displays preview of the validate dataset after subcatagory conversion
unique_classes = validate_nslkdd['class'].unique()
print(f"{len(unique_classes)} unique classes:")
for cls in sorted(unique_classes, key=lambda x: str(x)):
    print(f"  {cls}: {(validate_nslkdd['class'] == cls).sum()}")
print("Preview of the first 5 instances of the Validate Dataset after subcatagory conversion: ")
validate_nslkdd.head()

3 unique classes:
  DoS: 4594
  Normal: 6734
  Probe: 1165
Preview of the first 5 instances of the Validate Dataset after subcatagory conversion: 


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class
0,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0.0,0.0,1.0,1.0,1.00,0.00,0.00,27,2,0.04,1.00,0.04,1.00,0.0,0.0,0.96,1.0,Probe
1,0,tcp,http,SF,326,328,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,5,5,0.0,0.0,0.0,0.0,1.00,0.00,0.00,5,255,1.00,0.00,0.20,0.04,0.0,0.0,0.00,0.0,Normal
2,0,tcp,nnsp,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,127,5,1.0,1.0,0.0,0.0,0.04,0.07,0.00,255,5,0.02,0.07,0.00,0.00,1.0,1.0,0.00,0.0,DoS
3,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,222,11,0.0,0.0,1.0,1.0,0.05,0.06,0.00,255,11,0.04,0.06,0.00,0.00,0.0,0.0,1.00,1.0,DoS
4,0,udp,domain_u,SF,42,42,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,134,214,0.0,0.0,0.0,0.0,1.00,0.00,0.01,255,255,1.00,0.00,0.01,0.00,0.0,0.0,0.00,0.0,Normal


In [ ]:
# Export the validate dataset after subcatagory conversion
validate_nslkdd.to_csv('/content/drive/MyDrive/xg-ids/3class/datasets/3class_VALIDATE.txt', index=False)
print(f"Saved. Total instances: {validate_nslkdd.shape[0]}")

Saved. Total instances: 12493


#~~~~~ TRAIN DATASET PROCESSING ~~~~~

---



In [ ]:
# Load in the train dataset (with headers)
train_nslkdd = pd.read_csv('/content/drive/MyDrive/xg-ids/3class/datasets/3class_TRAIN.txt')

# Displays preview of the train dataset
print(f"Total instances: {train_nslkdd.shape[0]}")
print(f"Total Columns (41 features, 1 class label): {train_nslkdd.shape[1]}")
print("Preview of the first 5 instances of the Train Dataset before processing:")
train_nslkdd.head()

Total instances: 112433
Total Columns (41 features, 1 class label): 42
Preview of the first 5 instances of the Train Dataset before processing:


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class
0,3075,tcp,telnet,SF,1481,7668,0,0,0,0,1,1,6,0,0,0,1,0,0,0,0,0,1,1,0.0,0.0,0.0,0.0,1.00,0.00,0.00,255,2,0.01,0.06,0.00,0.00,0.95,0.0,0.0,0.0,normal
1,0,tcp,http,SF,238,17068,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,11,41,0.0,0.0,0.0,0.0,1.00,0.00,0.15,11,255,1.00,0.00,0.09,0.01,0.00,0.0,0.0,0.0,normal
2,0,icmp,ecr_i,SF,1480,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,3,0.0,0.0,0.0,0.0,1.00,0.00,0.67,1,39,1.00,0.00,1.00,0.51,0.00,0.0,0.0,0.0,pod
3,0,tcp,efs,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,102,2,0.0,0.0,1.0,1.0,0.02,0.06,0.00,255,2,0.01,0.06,0.00,0.00,0.00,0.0,1.0,1.0,neptune
4,0,tcp,http,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,2,1.0,1.0,0.0,0.0,0.67,0.67,0.00,245,62,0.25,0.02,0.00,0.00,0.99,1.0,0.0,0.0,neptune


In [ ]:
# Balance Training Dataset
keep_neptune = train_nslkdd[train_nslkdd['class'] == 'neptune'].sample(n=9759, random_state=42)
keep_normal  = train_nslkdd[train_nslkdd['class'] == 'normal'].sample(n=14000, random_state=42)
keep_rest    = train_nslkdd[~train_nslkdd['class'].isin(['neptune', 'normal'])]

nslkdd_balanced = pd.concat([keep_neptune, keep_normal, keep_rest]).reset_index(drop=True)

print(f"Total instances after balancing: {nslkdd_balanced.shape[0]}")
print("\nSubcategory distribution:")
print(nslkdd_balanced['class'].value_counts())


Total instances after balancing: 38491

Subcategory distribution:
class
normal       14000
neptune       9759
satan         3270
ipsweep       3239
portsweep     2638
smurf         2381
nmap          1344
back           860
teardrop       803
pod            181
land            16
Name: count, dtype: int64


In [ ]:
# Save balanced train dataset before major category conversion (subcategory labels preserved)
nslkdd_balanced.to_csv('/content/drive/MyDrive/xg-ids/3class/datasets/3class_TRAIN.txt', index=False)
print(f"Saved 3class_TRAIN.txt Total instances: {nslkdd_balanced.shape[0]}")

Saved 3class_TRAIN.txt Total instances: 38491


In [ ]:
# Rename subcategories to major categories
nslkdd_balanced['class'] = nslkdd_balanced['class'].map(subcategory_to_major)

unique_classes = nslkdd_balanced['class'].unique()
print(f"{len(unique_classes)} unique classes:")
for cls in sorted(unique_classes, key=lambda x: str(x)):
    print(f"  {cls}: {(nslkdd_balanced['class'] == cls).sum()}")
print("Preview of the first 5 instances of the Train Dataset after subcategory conversion: ")
nslkdd_balanced.head()

3 unique classes:
  DoS: 14000
  Normal: 14000
  Probe: 10491
Preview of the first 5 instances of the Train Dataset after subcategory conversion: 


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class
0,0,tcp,ssh,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,263,19,0.0,0.0,1.0,1.0,0.07,0.06,0.0,255,19,0.07,0.07,0.0,0.0,0.0,0.0,1.0,1.0,DoS
1,0,tcp,daytime,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,264,8,1.0,1.0,0.0,0.0,0.03,0.06,0.0,255,8,0.03,0.05,0.0,0.0,1.0,1.0,0.0,0.0,DoS
2,0,tcp,klogin,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,70,5,1.0,1.0,0.0,0.0,0.07,0.06,0.0,255,5,0.02,0.05,0.0,0.0,1.0,1.0,0.0,0.0,DoS
3,0,tcp,private,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,144,5,1.0,1.0,0.0,0.0,0.03,0.06,0.0,255,5,0.02,0.05,0.0,0.0,1.0,1.0,0.0,0.0,DoS
4,0,tcp,netstat,S0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,143,14,1.0,1.0,0.0,0.0,0.10,0.06,0.0,255,8,0.03,0.07,0.0,0.0,1.0,1.0,0.0,0.0,DoS


In [ ]:
nslkdd_balanced.to_csv('/content/drive/MyDrive/xg-ids/3class/datasets/3class_TRAIN.txt', index=False)
print(f"Export complete. Total instances: {nslkdd_balanced.shape[0]}")

Export complete. Total instances: 38491


#~~~~~ TEST DATASET PROCESSING ~~~~~


In [ ]:
# Load in the Test dataset + Apply Column Headers
test_nslkdd = pd.read_csv('/content/drive/MyDrive/xg-ids/raw_datasets/KDDTest.txt', header=None, names=column_names)

# Displays preview of the Test dataset
print(f"Total instances: {test_nslkdd.shape[0]}")
print(f"Total Columns (41 features, 1 class label, and 1 difficulty ranking): {test_nslkdd.shape[1]}")
print("Preview of the first 5 instances of the NSL-KDD Test Dataset before processing:")
test_nslkdd.head()

Total instances: 22544
Total Columns (41 features, 1 class label, and 1 difficulty ranking): 43
Preview of the first 5 instances of the NSL-KDD Test Dataset before processing:


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class,difficulty
0,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,229,10,0.0,0.00,1.0,1.0,0.04,0.06,0.00,255,10,0.04,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21
1,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,136,1,0.0,0.00,1.0,1.0,0.01,0.06,0.00,255,1,0.00,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0.0,0.00,0.0,0.0,1.00,0.00,0.00,134,86,0.61,0.04,0.61,0.02,0.0,0.0,0.00,0.00,normal,21
3,0,icmp,eco_i,SF,20,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,65,0.0,0.00,0.0,0.0,1.00,0.00,1.00,3,57,1.00,0.00,1.00,0.28,0.0,0.0,0.00,0.00,saint,15
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,8,0.0,0.12,1.0,0.5,1.00,0.00,0.75,29,86,0.31,0.17,0.03,0.02,0.0,0.0,0.83,0.71,mscan,11


In [ ]:
# Dropping difficulty column (last column) from test dataset
test_nslkdd = test_nslkdd.drop('difficulty', axis=1)
# Displays preview of the validate dataset after dropping difficulty column
print(f"Total Columns (41 features and 1 class label: {test_nslkdd.shape[1]}")
print("Preview of the first 5 instances of the NSL-KDD Dataset after dropping difficulty column:")
test_nslkdd.head()

Total Columns (41 features and 1 class label: 42
Preview of the first 5 instances of the NSL-KDD Dataset after dropping difficulty column:


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class
0,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,229,10,0.0,0.00,1.0,1.0,0.04,0.06,0.00,255,10,0.04,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune
1,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,136,1,0.0,0.00,1.0,1.0,0.01,0.06,0.00,255,1,0.00,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0.0,0.00,0.0,0.0,1.00,0.00,0.00,134,86,0.61,0.04,0.61,0.02,0.0,0.0,0.00,0.00,normal
3,0,icmp,eco_i,SF,20,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,65,0.0,0.00,0.0,0.0,1.00,0.00,1.00,3,57,1.00,0.00,1.00,0.28,0.0,0.0,0.00,0.00,saint
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,8,0.0,0.12,1.0,0.5,1.00,0.00,0.75,29,86,0.31,0.17,0.03,0.02,0.0,0.0,0.83,0.71,mscan


In [ ]:
# Remove R2L and U2R

# Identify subcategories that belong to U2R or R2L
drop_subcategories = {k for k, v in subcategory_to_major.items() if v in ('U2R', 'R2L')}

# Drop all rows where 'class' contains subcategories belonging to U2R or R2L
test_nslkdd = test_nslkdd[~test_nslkdd['class'].isin(drop_subcategories)].reset_index(drop=True)

print(f"Total instances after dropping U2R + R2L: {test_nslkdd.shape[0]}")
print("Remaining subcategories:", sorted(test_nslkdd['class'].unique()))

Total instances after dropping U2R + R2L: 19590
Remaining subcategories: ['apache2', 'back', 'ipsweep', 'land', 'mailbomb', 'mscan', 'neptune', 'nmap', 'normal', 'pod', 'portsweep', 'processtable', 'saint', 'satan', 'smurf', 'teardrop', 'udpstorm']


In [ ]:
# Rename subcategories to major categories
test_nslkdd['class'] = test_nslkdd['class'].map(subcategory_to_major)


# Displays preview of the test dataset after subcatagory conversion
unique_classes = test_nslkdd['class'].unique()
print(f"{len(unique_classes)} unique classes:")
for cls in sorted(unique_classes, key=lambda x: str(x)):
    print(f"  {cls}: {(test_nslkdd['class'] == cls).sum()}")
print("Preview of the first 5 instances of the Test Dataset after subcatagory conversion: ")
test_nslkdd.head()

3 unique classes:
  DoS: 7458
  Normal: 9711
  Probe: 2421
Preview of the first 5 instances of the Test Dataset after subcatagory conversion: 


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class
0,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,229,10,0.0,0.00,1.0,1.0,0.04,0.06,0.00,255,10,0.04,0.06,0.00,0.00,0.0,0.0,1.00,1.00,DoS
1,0,tcp,private,REJ,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,136,1,0.0,0.00,1.0,1.0,0.01,0.06,0.00,255,1,0.00,0.06,0.00,0.00,0.0,0.0,1.00,1.00,DoS
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0.0,0.00,0.0,0.0,1.00,0.00,0.00,134,86,0.61,0.04,0.61,0.02,0.0,0.0,0.00,0.00,Normal
3,0,icmp,eco_i,SF,20,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,65,0.0,0.00,0.0,0.0,1.00,0.00,1.00,3,57,1.00,0.00,1.00,0.28,0.0,0.0,0.00,0.00,Probe
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,8,0.0,0.12,1.0,0.5,1.00,0.00,0.75,29,86,0.31,0.17,0.03,0.02,0.0,0.0,0.83,0.71,Probe


In [ ]:
# Export the test dataset after subcatagory conversion
test_nslkdd.to_csv('/content/drive/MyDrive/xg-ids/3class/datasets/3class_test.txt', index=False)
print(f"Saved. Total instances: {test_nslkdd.shape[0]}")

Saved. Total instances: 19590
